In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
df = pd.read_csv("cost_estimation_dataset.csv")
df.head()

,case_type,jurisdiction,location,complexity,duration_months,lawyer_level,legal_fees,court_fees,filing_admin_fees,miscellaneous_fees
0,Civil,District Court,"Mumbai, Maharashtra",Medium,12,Mid-Senior,450000,50000,25000,30000
1,Criminal,High Court,"Delhi, Delhi",High,24,Senior Counsel,850000,120000,45000,55000
2,Corporate,Supreme Court,"Mumbai, Maharashtra",High,36,Top Tier Firm,2500000,200000,80000,120000
3,Family / Matrimonial,District Court,"Bangalore, Karnataka",Low,8,Junior Associate,150000,25000,15000,18000
4,Property Dispute,High Court,"Chennai, Tamil Nadu",Medium,18,Mid-Senior,550000,75000,35000,42000


In [3]:
df = pd.get_dummies(df, drop_first=True)
df.head()

,duration_months,legal_fees,court_fees,filing_admin_fees,miscellaneous_fees,case_type_Consumer,case_type_Corporate,case_type_Criminal,case_type_Family / Matrimonial,case_type_Property Dispute,...,"location_Vizianagaram, Andhra Pradesh","location_Warangal, Telangana","location_Wardha, Maharashtra","location_Yamunanagar, Haryana","location_Yavatmal, Maharashtra",complexity_Low,complexity_Medium,lawyer_level_Mid-Senior,lawyer_level_Senior Counsel,lawyer_level_Top Tier Firm
0,12,450000,50000,25000,30000,False,False,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,24,850000,120000,45000,55000,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
2,36,2500000,200000,80000,120000,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True
3,8,150000,25000,15000,18000,False,False,False,True,False,...,False,False,False,False,False,True,False,False,False,False
4,18,550000,75000,35000,42000,False,False,False,False,True,...,False,False,False,False,False,False,True,True,False,False


In [4]:
TARGET_FIELDS = [
    "legal_fees",
    "court_fees",
    "filing_admin_fees",
    "miscellaneous_fees",
]

X = df.drop(TARGET_FIELDS, axis=1)
y = df[TARGET_FIELDS]
X.head()

,duration_months,case_type_Consumer,case_type_Corporate,case_type_Criminal,case_type_Family / Matrimonial,case_type_Property Dispute,jurisdiction_District Court,jurisdiction_High Court,jurisdiction_Supreme Court,jurisdiction_Tribunal,...,"location_Vizianagaram, Andhra Pradesh","location_Warangal, Telangana","location_Wardha, Maharashtra","location_Yamunanagar, Haryana","location_Yavatmal, Maharashtra",complexity_Low,complexity_Medium,lawyer_level_Mid-Senior,lawyer_level_Senior Counsel,lawyer_level_Top Tier Firm
0,12,False,False,False,False,False,True,False,False,False,...,False,False,False,False,False,False,True,True,False,False
1,24,False,False,True,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
2,36,False,True,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
3,8,False,False,False,True,False,True,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,18,False,False,False,False,True,False,True,False,False,...,False,False,False,False,False,False,True,True,False,False


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

MAE: 7278.975373482724
R2: 0.996112117873923


In [6]:
import numpy as np

# Sample input (before one-hot encoding)
sample = pd.DataFrame([{
    "case_type": "Criminal",
    "jurisdiction": "High Court",
    "location": "Delhi, Delhi",
    "complexity": "High",
    "duration_months": 24,
    "lawyer_level": "Senior Counsel",
}])

# Apply same encoding as training data
sample_encoded = pd.get_dummies(sample, drop_first=True)

# Align columns with training features (fill missing columns with 0)
sample_encoded = sample_encoded.reindex(columns=X.columns, fill_value=0)

# Predict
prediction = model.predict(sample_encoded)[0]

# Build result
result = dict(zip(TARGET_FIELDS, prediction))
result["total_cost"] = sum(result.values())

for key, value in result.items():
    print(f"{key}: ₹{value:,.2f}")

legal_fees: ₹885,400.00
court_fees: ₹114,776.67
filing_admin_fees: ₹50,623.33
miscellaneous_fees: ₹64,380.00
total_cost: ₹1,115,180.00
